### TÓM TẮT THỰC NGHIỆM TÁI TẠO COSTA (2002)
*(Mục tiêu: Kiểm chứng tính chính xác của framework `core_ntsa/mse.py`)*

#### 1. Thiết lập & Chuẩn bị (What to do)
*   **Tham số hệ thống**:
    *   Chiều dài dữ liệu: N = 30,000
    *   Chiều vector nhúng: m = 2
    *   Dung sai: r = 0.15 × SD (bắt buộc khóa cứng SD của chuỗi gốc ở mọi scale)
    *   Thang đo: $\tau$ quét từ 1 đến 20.
*   **3 Tập dữ liệu mô phỏng**:
    *   **Chuỗi A (White Noise)**: Nhiễu trắng Gaussian (hệ thống ngẫu nhiên, không liên kết).
    *   **Chuỗi B (1/f Noise)**: Nhiễu $1/f$ (hệ thống phức tạp tự đồng dạng, tạo qua biến đổi FFT/IFFT).
    *   **Chuỗi C (Surrogate)**: Dữ liệu xáo trộn ngẫu nhiên từ Chuỗi B để phá hủy cấu trúc tương quan dài hạn.
*   **Luồng xử lý (Pipeline)**: Chạy qua 2 giai đoạn của MSE (Coarse-Graining + SampEn). *Lưu ý hiệu năng:* Tối ưu hóa lõi đếm của SampEn bằng cấu trúc `cKDTree` (khoảng cách Chebyshev) để đạt tốc độ thực thi cao.

#### 2. Kết quả Kỳ vọng (Expected Outcomes)
*   **Tái tạo Hình 1 (White Noise vs. 1/f Noise)**:
    *   **Nhiễu trắng (A)**: Đường SampEn sụt giảm mạnh từ mức $\approx$ 2.47 (ở $\tau$ = 1) xuống sát 1.0 (ở $\tau$ = 20) do phép trung bình hóa triệt tiêu nhiễu ngẫu nhiên.
    *   **Nhiễu 1/f (B)**: Đường SampEn duy trì ổn định hoàn hảo quanh mức 1.8 - 1.9 trên toàn bộ dải thang đo $\tau$.
    *   **Điểm giao cắt**: Đồ thị của Chuỗi A và Chuỗi B phải cắt nhau chính xác tại thang đo $\tau$ = 5.
*   **Tái tạo Hình 2 (Surrogate Test)**:
    *   **Chuỗi Surrogate (C)**: Đồ thị MSE sẽ sụp đổ hoàn toàn, mang hình dáng cắm xuống y hệt như nhiễu trắng (Chuỗi A). Điều này chứng minh phép xáo trộn đã phá hủy bản chất động học phức tạp của tín hiệu $1/f$.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from typing import Optional
sys.path.append(os.path.abspath(".."))
from core_ntsa.mse import mse